# Regional Fine-Tuning of the Pilot Models

This notebook tests whether the models pre-trained on the global grid at 5.6 degrees can be adapted to a regional grid at a finer resolution. The region is a patch of 16 x 16 points at 1.5 degrees over Pakistan and northern India (18.0 to 40.5 degrees north, 58.5 to 81.0 degrees east), taken from ERA5 in WeatherBench 2. The patch is normalised with the statistics of the global pre-training data, so that the pre-trained models see inputs on the scale they were trained on.

Every pre-trained model is evaluated twice under the same protocols: once directly on the patch without any training (zero-shot), and once after fine-tuning for 5,000 steps (`configs/train/finetune.yaml`, the pilot settings otherwise, seed 42).

- **WIND** is adapted by full fine-tuning, because it has no mechanism to adapt only a part of itself. All learned weights fit the patch size, and the rotary position embeddings are rebuilt for it. The sizes S, M and L start from the 22,500-step milestones of the main pilot.
- **SpherIDiff** is adapted through the Slepian basis of the patch. With the band limit L' = 61 of the 1.5 degree grid, the Shannon number of the patch is M = 45. In each block, the pre-trained spectral weight is restricted to these modes, G0 = U_M^T W0 U_M, and kept frozen, and a trainable M x M update dG is added, which starts at zero. The pointwise weights (uplift, channel MLPs, temporal attention, downlift) are fine-tuned as well. The b4 model at L = 16 starts from its 22,500-step milestone, and the four models at L = 31 start from their final 25,000-step weights. Two further runs train only dG and keep everything else frozen.

| Run | Start | Adaptation | Trainable parameters |
|---|---|---|---|
| `finetune_wind_s` | WIND-S, 22,500 steps | full | 3,005,369 |
| `finetune_wind_m` | WIND-M, 22,500 steps | full | 7,336,121 |
| `finetune_wind_l` | WIND-L, 22,500 steps | full | 12,238,201 |
| `finetune_spheridiff_b4_l16` | b4 at L = 16, 22,500 steps | dG and pointwise | 1,347,517 |
| `finetune_spheridiff_l31_xs` | xs at L = 31, 25,000 steps | dG and pointwise | 144,107 |
| `finetune_spheridiff_l31_s` | s at L = 31, 25,000 steps | dG and pointwise | 94,141 |
| `finetune_spheridiff_l31_m` | m at L = 31, 25,000 steps | dG and pointwise | 346,045 |
| `finetune_spheridiff_l31_b4` | b4 at L = 31, 25,000 steps | dG and pointwise | 1,347,517 |
| `finetune_spheridiff_b4_l16_delta_only` | b4 at L = 16, 22,500 steps | dG only | 8,100 |
| `finetune_spheridiff_l31_b4_delta_only` | b4 at L = 31, 25,000 steps | dG only | 8,100 |

The notebook is designed for a background run on two T4 GPUs. It needs two Kaggle inputs, which Kaggle extracts on upload: the pre-trained weights (folders `pilot` and `pilot_l31`) and the regional data cache (folder `era5_1p5deg_south_asia`). The repository code is cloned from GitHub. The results are packed into `finetune_results.zip` in the working folder.

In [ ]:
import os
import sys
from pathlib import Path

import torch

INPUT_ROOT = Path("/kaggle/input")
WORK_ROOT = Path("/kaggle/working")
RUNS_ROOT = WORK_ROOT / "runs"
FINAL_PROTOCOL = "final_v1"
ZERO_SHOT_SUFFIX = "_zeroshot"
WIND_EXPERIMENTS = ["finetune_wind_s", "finetune_wind_m", "finetune_wind_l"]
SPHERIDIFF_EXPERIMENTS = [
    "finetune_spheridiff_l31_xs",
    "finetune_spheridiff_l31_s",
    "finetune_spheridiff_l31_m",
    "finetune_spheridiff_l31_b4",
    "finetune_spheridiff_b4_l16",
    "finetune_spheridiff_l31_b4_delta_only",
    "finetune_spheridiff_b4_l16_delta_only",
]
DELTA_ONLY = [name for name in SPHERIDIFF_EXPERIMENTS if name.endswith("_delta_only")]
ZERO_SHOT_EXPERIMENTS = WIND_EXPERIMENTS + [name for name in SPHERIDIFF_EXPERIMENTS if name not in DELTA_ONLY]
RUN_NAMES = WIND_EXPERIMENTS + SPHERIDIFF_EXPERIMENTS
ZERO_SHOT_NAMES = [name + ZERO_SHOT_SUFFIX for name in ZERO_SHOT_EXPERIMENTS]
ARCHIVE = WORK_ROOT / "finetune_results.zip"

## Inputs and hardware

The repository code is cloned from GitHub (`https://github.com/syeddaniyalg/spheri-diff`) into the working folder, which requires internet access for the notebook. The regional data cache is the folder under `/kaggle/input` that holds `stats.npz`, and the pre-trained weights are the folder that holds both `pilot` and `pilot_l31`. Both are found by searching `/kaggle/input`, so the names that Kaggle gives to the uploaded datasets do not matter.

In [ ]:
import subprocess

REPOSITORY_URL = "https://github.com/syeddaniyalg/spheri-diff"
CODE_ROOT = WORK_ROOT / "spheri-diff"


def locate(marker, root):
    for current, _, files in os.walk(root):
        if marker in files:
            return Path(current)
    raise FileNotFoundError(f"{marker} was not found in {root}")


def locate_pretrained(root):
    for current, folders, _ in os.walk(root):
        if "pilot" in folders and "pilot_l31" in folders:
            return Path(current)
    raise FileNotFoundError(f"no folder with pilot and pilot_l31 was found in {root}")


if not CODE_ROOT.exists():
    subprocess.run(["git", "clone", "--depth", "1", REPOSITORY_URL, str(CODE_ROOT)], check=True)
CACHE_ROOT = locate("stats.npz", INPUT_ROOT)
PRETRAINED_ROOT = locate_pretrained(INPUT_ROOT)
sys.path.insert(0, str(CODE_ROOT / "src"))
print("code:", CODE_ROOT)
print("data:", CACHE_ROOT)
print("pretrained weights:", PRETRAINED_ROOT)
for index in range(torch.cuda.device_count()):
    print("gpu", index, torch.cuda.get_device_name(index))

## Running the jobs

The first GPU runs the WIND jobs only and the second GPU runs the SpherIDiff jobs only, each as one queue of separate processes. A queue first fine-tunes its models one after another, and then evaluates every model under the fixed protocol `final_v1` on the test year 2021 of the patch: the forecast error and CRPS, and the six reconstruction tasks with MMPS guidance and two forms of native denoising. Each pre-trained model is evaluated once without training (the run folders ending in `_zeroshot`, whose weights the fine-tuning job saves before its first step) and every fine-tuned model once after training. With one GPU, both queues share it.

During fine-tuning, every run is evaluated every 250 steps on 32 fixed validation windows of 2020, and the evaluation at step 0 is the zero-shot result on the same windows. Each job has its own log file in `runs/_logs`. The notebook prints every line that a job writes, prefixed with the elapsed time, the GPU and the job name, and keeps the same lines in `runs/_logs/parallel.log`.

The jobs and the packing of the results share one cell with a `try` and a `finally`, so the figures, the status file and the archive are produced even when a job fails or the cell is interrupted. The archive is also refreshed every 30 minutes during the run.

In [ ]:
from spheridiff.kaggle.finalise import finalise, refresh_archive
from spheridiff.kaggle.jobs import final_job, finetune_job, place_queues
from spheridiff.kaggle.runner import ParallelRunner, RunContext

GPUS = torch.cuda.device_count()
CONTEXT = RunContext(CODE_ROOT, RUNS_ROOT, RUNS_ROOT / "_logs", CACHE_ROOT)


def queue(experiments):
    training = [finetune_job(name, PRETRAINED_ROOT) for name in experiments]
    zero_shot = [
        final_job(name, FINAL_PROTOCOL, name + ZERO_SHOT_SUFFIX) for name in experiments if name in ZERO_SHOT_EXPERIMENTS
    ]
    fine_tuned = [final_job(name, FINAL_PROTOCOL) for name in experiments]
    return training + zero_shot + fine_tuned


PLACEMENT = [(0, queue(WIND_EXPERIMENTS)), (1, queue(SPHERIDIFF_EXPERIMENTS))]


def refresh():
    return refresh_archive(RUNS_ROOT, ARCHIVE)


RUNNER = None
try:
    RUNNER = ParallelRunner(place_queues(PLACEMENT, GPUS), CONTEXT, refresh=refresh)
    RUNNER.run()
finally:
    OUTCOMES = RUNNER.outcomes() if RUNNER else []
    finalise(RUNS_ROOT, ARCHIVE, OUTCOMES, RUN_NAMES, final_names=ZERO_SHOT_NAMES + RUN_NAMES)
print(OUTCOMES)
print(ARCHIVE, f"{ARCHIVE.stat().st_size / 1024**2:.1f} MB")

## Results

The summaries list the steps, training time, time per step, peak memory and final loss of every fine-tuning run. The final evaluation table lists every zero-shot model (rows ending in `_zeroshot`, with 0 trained steps) and every fine-tuned model under the same protocol: the number of trainable parameters, the trained steps, the forecast error and CRPS, and for each task and method the normalised error on the unobserved entries, with the reference of the task as the last row. The figures show the validation error of all runs against fine-tuning steps, samples seen and training time, starting from the zero-shot value at step 0.

In [ ]:
import json

for name in RUN_NAMES:
    summary = RUNS_ROOT / name / "summary.json"
    print(name, json.loads(summary.read_text()) if summary.exists() else "no summary, see runs/_logs")

final_summary = RUNS_ROOT / "final_evaluation_summary.txt"
if final_summary.exists():
    print(final_summary.read_text(encoding="utf-8"))

In [ ]:
from IPython.display import Image, display

for figure in ["evaluation_curves.png", "training_loss.png"]:
    path = RUNS_ROOT / figure
    if path.exists():
        display(Image(filename=str(path)))